> **Solutions for `6-calculus-lab.ipynb`.** Work through the lab on your own first. Open this notebook to check your answers after you have made a real attempt, or when you have been stuck on a part for a while. Many questions have more than one good answer, so compare your reasoning, not just your wording.

# Lab 6: Calculus and Gradient Descent

> **Course:** Data Science Foundations
> **Lesson:** Day 6 Lab, companion to `6-calculus-essentials.md`
> **Dataset:** `mpg` (seaborn), the same regression problem as Lab 5
> **Estimated time:** 2 hours

## Lab Objectives

- Compare forward and central finite differences, and see why $h$ can be *too* small
- Differentiate exponentials, logarithms and the **sigmoid**, and check each numerically
- Compute partial derivatives and verify a gradient with a **gradient check**
- Implement gradient descent in 1-D and explain exactly when it converges or diverges
- See gradient descent get stuck in a **local minimum** on a non-convex function
- Fit linear regression by gradient descent with a proper stopping rule, and link the maximum learning rate to linear algebra
- Implement **mini-batch stochastic gradient descent**

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

## Part 1: How Small Should $h$ Be?

The lesson defines the derivative with a **forward** difference $\frac{f(x+h) - f(x)}{h}$ but its code uses a **central** difference $\frac{f(x+h) - f(x-h)}{2h}$. Find out why.

In [ ]:
def forward_diff(f, x, h):
    return (f(x + h) - f(x)) / h


def central_diff(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)


hs = 10.0 ** -np.arange(1, 13)
exact = np.cos(1.0)
err_fwd = [abs(forward_diff(np.sin, 1.0, h) - exact) for h in hs]
err_cen = [abs(central_diff(np.sin, 1.0, h) - exact) for h in hs]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(hs, err_fwd, "o-", label="forward difference")
ax.loglog(hs, err_cen, "s-", label="central difference")
ax.set_xlabel("h")
ax.set_ylabel("absolute error")
ax.invert_xaxis()
ax.set_title("Finite-difference error for d/dx sin(x) at x = 1")
ax.legend()
plt.show()

print(pd.DataFrame({"h": hs, "forward": err_fwd, "central": err_cen}).to_string(index=False))

**Question 1.** Describe the two curves. Why does the error go *up* again when $h$ becomes very small? What $h$ would you use for each method?

**Model answer:**

Moving right to left (smaller $h$), both errors first fall in a straight line on the log-log plot. The forward error shrinks in proportion to $h$; the central error shrinks in proportion to $h^2$, twice as steeply, because its errors on the two sides cancel. Then both turn around and grow. Computers store about 16 significant digits, so $f(x+h) - f(x)$ subtracts two nearly identical numbers and the rounding error gets divided by a tiny $h$ (catastrophic cancellation). The best $h$ is around $10^{-8}$ for forward differences and $10^{-5}$ to $10^{-6}$ for central ones, which is why the lesson's `h=1e-5` works well.

## Part 2: Exponentials, Logarithms and the Sigmoid

The lesson's rule table has no $e^x$ or $\ln x$, yet these appear in almost every ML loss function. Add them:

| Function | Derivative |
|---|---|
| $e^x$ | $e^x$ |
| $\ln x$ | $1/x$ |
| $e^{g(x)}$ (chain rule) | $g'(x)\,e^{g(x)}$ |
| $\sigma(x) = \frac{1}{1 + e^{-x}}$ (sigmoid, used in logistic regression) | $\sigma(x)\,(1 - \sigma(x))$ |

For each function below, work out the derivative **by hand**, code it, and let the checkpoint compare it with `central_diff`.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


problems = {
    "3x^4 - 2x + 5": (lambda x: 3 * x**4 - 2 * x + 5, lambda x: 12 * x**3 - 2),
    "e^(2x)":        (lambda x: np.exp(2 * x),        lambda x: 2 * np.exp(2 * x)),
    "ln(x^2 + 1)":   (lambda x: np.log(x**2 + 1),     lambda x: 2 * x / (x**2 + 1)),
    "x e^(-x)":      (lambda x: x * np.exp(-x),       lambda x: np.exp(-x) * (1 - x)),
    "sigmoid(x)":    (sigmoid,                        lambda x: sigmoid(x) * (1 - sigmoid(x))),
}

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
x_test = np.linspace(-2, 2, 41)
for name, (f, df) in problems.items():
    numeric = central_diff(f, x_test, 1e-5)
    assert df(x_test) is not None, f"{name}: derivative not implemented"
    rel_err = np.max(np.abs(df(x_test) - numeric) / (1 + np.abs(numeric)))
    assert rel_err < 1e-6, f"{name}: max relative error {rel_err:.2e}"
    print(f"{name:<15s} OK (max rel. error {rel_err:.1e})")
print("passed")

In [ ]:
x = np.linspace(-8, 8, 400)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, sigmoid(x), lw=2, label="sigmoid(x)")
ax.plot(x, sigmoid(x) * (1 - sigmoid(x)), lw=2, label="sigmoid'(x)")
ax.legend()
ax.set_title("The sigmoid and its derivative (max 0.25 at x = 0)")
plt.show()

**Model answer:**

The derivative peaks at $x = 0$ with value $0.5 \times 0.5 = 0.25$ and falls towards 0 in both tails. When a sigmoid unit is saturated (very large $|x|$), its gradient is almost zero, so gradient descent barely updates it. This is the "vanishing gradient" problem you will meet in neural networks.

## Part 3: Partial Derivatives and the Gradient Check

$$f(x, y) = x^2 y + 3y^3 - 2x$$

When you code a gradient by hand, the standard safety net is a **gradient check**: compare it with a numerical gradient, one coordinate at a time.

In [ ]:
def f(p):
    x, y = p
    return x**2 * y + 3 * y**3 - 2 * x


def grad_f(p):
    x, y = p
    return np.array([2 * x * y - 2, x**2 + 9 * y**2])


def numerical_gradient(func, p, h=1e-5):
    p = np.asarray(p, dtype=float)
    grad = np.zeros_like(p)
    for i in range(len(p)):
        step = np.zeros_like(p)
        step[i] = h
        grad[i] = (func(p + step) - func(p - step)) / (2 * h)
    return grad

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
for p in rng.normal(size=(5, 2)) * 3:
    assert np.allclose(grad_f(p), numerical_gradient(f, p), rtol=1e-5, atol=1e-6), p
assert np.allclose(grad_f([1, 2]), [2, 37])
print("passed")

In [ ]:
def g(x, y):
    return (x - 1) ** 2 + 2 * (y + 0.5) ** 2

xs, ys = np.meshgrid(np.linspace(-2, 4, 200), np.linspace(-3, 2, 200))
qx, qy = np.meshgrid(np.linspace(-2, 4, 13), np.linspace(-3, 2, 11))
gx, gy = 2 * (qx - 1), 4 * (qy + 0.5)

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contour(xs, ys, g(xs, ys), levels=15, cmap="viridis")
ax.quiver(qx, qy, -gx, -gy, color="tomato", angles="xy")
ax.plot(1, -0.5, "k*", ms=15, label="minimum")
ax.set_aspect("equal")
ax.set_title("Negative gradient: downhill and perpendicular to the contours")
ax.legend()
plt.show()

**Model answer:**

Every arrow points downhill towards the minimum at $(1, -0.5)$ and crosses the contour lines at right angles. Arrows are longer far from the minimum (steeper slope) and shrink near it. Note `set_aspect("equal")`: without it, unequal axis scales make the arrows *look* non-perpendicular. The lesson's gradient descent path plot in Section 6.8 uses the same line for the same reason.

## Part 4: Gradient Descent in One Dimension

$$f(x) = (x - 3)^2 + 2, \qquad f'(x) = 2(x - 3)$$

In [ ]:
def f1(x):
    return (x - 3) ** 2 + 2


def df1(x):
    return 2 * (x - 3)


def gradient_descent_1d(df, x0, lr, n_iters):
    xs = [x0]
    for _ in range(n_iters):
        xs.append(xs[-1] - lr * df(xs[-1]))
    return np.array(xs)


fig, ax = plt.subplots(figsize=(9, 4.5))
for lr in [0.01, 0.1, 0.5, 0.9, 1.05]:
    path = gradient_descent_1d(df1, -2.0, lr, 30)
    ax.plot(path, "o-", ms=3, label=f"lr = {lr}  (final x = {path[-1]:.3g})")
ax.axhline(3, color="black", ls=":", label="minimum x = 3")
ax.set_ylim(-10, 16)
ax.set_xlabel("iteration t")
ax.set_ylabel("x_t")
ax.legend(fontsize=8)
plt.show()

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
path = gradient_descent_1d(df1, -2.0, 0.1, 100)
assert len(path) == 101 and abs(path[-1] - 3) < 1e-6
print("passed")

**Question 4.** Substitute the update into itself: show that $x_{t+1} - 3 = (1 - 2\alpha)(x_t - 3)$. Use it to explain every curve you plotted, and find the exact range of learning rates for which gradient descent converges.

**Model answer:**

$x_{t+1} = x_t - \alpha \cdot 2(x_t - 3)$, so $x_{t+1} - 3 = (x_t - 3) - 2\alpha(x_t - 3) = (1 - 2\alpha)(x_t - 3)$. The distance to the minimum is multiplied by $r = 1 - 2\alpha$ every step.

- α = 0.01: r = 0.98, slow crawl. After 30 steps x has only covered about 45% of the way to 3.
- α = 0.1: r = 0.8, smooth, fast convergence.
- α = 0.5: r = 0, lands exactly on the minimum in **one** step.
- α = 0.9: r = −0.8, overshoots and oscillates around 3, but still converges.
- α = 1.05: r = −1.1, each overshoot is bigger than the last: divergence.

It converges exactly when $|1 - 2\alpha| < 1$, i.e. $0 < \alpha < 1$. In general the limit is $\alpha < 2 / f''$. The curvature sets the maximum learning rate, which Part 6 extends to many dimensions.

## Part 5: Non-Convex Functions and Local Minima

Linear regression's loss is a convex bowl, so gradient descent always finds the global minimum. Most ML losses are not convex:

$$f(x) = x^4 - 3x^2 + x, \qquad f'(x) = 4x^3 - 6x + 1$$

In [ ]:
def f2(x):
    return x**4 - 3 * x**2 + x


def df2(x):
    return 4 * x**3 - 6 * x + 1


x = np.linspace(-2.2, 2.2, 400)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x, f2(x), color="black", lw=2)
for x0 in [-2, -0.5, 0.5, 2]:
    path = gradient_descent_1d(df2, float(x0), 0.01, 500)
    ax.plot(path[0], f2(path[0]), "o", ms=8)
    ax.annotate("", xy=(path[-1], f2(path[-1])), xytext=(path[0], f2(path[0])),
                arrowprops=dict(arrowstyle="->", lw=1.5))
    print(f"start {x0:+.1f} -> end x = {path[-1]:+.4f}, f = {f2(path[-1]):+.4f}")
ax.set_title("Where you start decides which minimum you find")
plt.show()

**Model answer:**

Starts at −2 and −0.5 roll into the **global** minimum near x ≈ −1.30 (f ≈ −3.51). Starts at 0.5 and 2 end in the **local** minimum near x ≈ 1.13 (f ≈ −1.07) and stay there, because the gradient is zero at the bottom of any valley. Gradient descent only ever sees the local slope. This is why deep learning uses random restarts, momentum, and noisy (stochastic) gradients, and why convexity makes linear regression so well-behaved.

## Part 6: Linear Regression by Gradient Descent, with a Stopping Rule

The lesson runs a fixed 1,000 iterations. A better loop stops when the gradient is tiny: $\|\nabla L\| < \text{tol}$. Same data preparation as Lab 5 (split first, then scale).

In [ ]:
cars = sns.load_dataset("mpg").dropna().reset_index(drop=True)
predictors = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
X = cars[predictors].to_numpy(dtype=float)
y = cars["mpg"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
Xd_train = np.hstack([np.ones((len(X_train), 1)), scaler.transform(X_train)])
Xd_test = np.hstack([np.ones((len(X_test), 1)), scaler.transform(X_test)])
w_exact = np.linalg.solve(Xd_train.T @ Xd_train, Xd_train.T @ y_train)
print("Normal-equation weights:", w_exact.round(3))

In [ ]:
def gd_linear_regression(X, y, lr, tol=1e-6, max_iters=200_000):
    m, n = X.shape
    w = np.zeros(n)
    losses = []
    for t in range(max_iters):
        residuals = y - X @ w
        loss = np.mean(residuals ** 2)
        losses.append(loss)
        if not np.isfinite(loss):
            break
        grad = -(2 / m) * (X.T @ residuals)
        if np.linalg.norm(grad) < tol:
            break
        w = w - lr * grad
    return w, losses, t + 1

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
w_gd, losses, n_used = gd_linear_regression(Xd_train, y_train, lr=0.05)
print(f"converged in {n_used} iterations; max |w_gd - w_exact| = {np.abs(w_gd - w_exact).max():.2e}")
assert np.allclose(w_gd, w_exact, atol=1e-4)
print("passed")

### 6.2 The maximum learning rate comes from linear algebra

The MSE loss has a constant second derivative (its **Hessian**) $H = \frac{2}{m} X^TX$. Generalising Part 4, gradient descent converges exactly when $\alpha < 2 / \lambda_{\max}$, where $\lambda_{\max}$ is the largest **eigenvalue** of $H$ (the curvature along the steepest direction of the bowl). The *smallest* eigenvalue sets how slowly the flattest direction converges.

In [ ]:
m = len(y_train)
eigvals = np.linalg.eigvalsh(2 / m * Xd_train.T @ Xd_train)
lr_max = 2 / eigvals.max()
print(f"Hessian eigenvalues: {eigvals.round(3)}")
print(f"lambda_max = {eigvals.max():.3f}  ->  lr_max = {lr_max:.4f}")
print(f"condition number lambda_max / lambda_min = {eigvals.max() / eigvals.min():.0f}")

fig, ax = plt.subplots(figsize=(9, 4.5))
rows = []
for lr in [0.01, 0.05, 0.1, 0.9 * lr_max, 1.02 * lr_max]:
    with np.errstate(over="ignore", invalid="ignore"):   # the diverging run overflows on purpose
        w_run, losses_run, n_run = gd_linear_regression(Xd_train, y_train, lr=lr, max_iters=20_000)
    converged = np.isfinite(losses_run[-1]) and np.allclose(w_run, w_exact, atol=1e-4)
    rows.append({"lr": round(lr, 4), "iterations": n_run, "final_loss": losses_run[-1], "converged": converged})
    ax.plot(np.clip(losses_run, None, 1e6), label=f"lr = {lr:.4f}")
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel("training MSE (clipped at 1e6)")
ax.legend()
plt.show()
pd.DataFrame(rows)

**Question 6.** What is `lr_max` for this problem? What happens just below and just above it? Why does even the best learning rate still need many iterations here? (Hint: look at the condition number and recall Lab 5's collinear features.)

**Model answer:**

`lr_max` = 2/λ_max ≈ 0.235 here (λ_max ≈ 8.5). Just below it (0.9 × lr_max) gradient descent converges, and it is the fastest of the rates tried. Just above it (1.02 × lr_max) the loss grows geometrically until it overflows to infinity: the steepest direction of the bowl overshoots by more each step, exactly like α = 1.05 in Part 4.

Even at the best rate, convergence along the *flattest* direction is slow. The Hessian's condition number is over 100 because `cylinders`, `displacement`, `horsepower` and `weight` are strongly collinear, so the bowl is a long, narrow valley. The step size is capped by the steep direction while progress is limited by the flat one. This is why feature scaling matters, why a fixed 1,000 iterations only got the lesson's California Housing run to about two decimal places, and why optimisers like momentum and Adam exist.

## Part 7: Mini-Batch Stochastic Gradient Descent (extension)

Full-batch GD computes the gradient over all $m$ rows for every step. **Mini-batch SGD** shuffles the data each **epoch** (one pass through it) and takes one step per batch of $b$ rows, using that batch's gradient as a cheap, noisy estimate. The lesson mentions SGD; here you build it.

In [ ]:
def minibatch_sgd(X, y, lr, batch_size, n_epochs, rng=rng):
    m, n = X.shape
    w = np.zeros(n)
    epoch_losses = []
    for _ in range(n_epochs):
        order = rng.permutation(m)
        for start in range(0, m, batch_size):
            idx = order[start:start + batch_size]
            Xb, yb = X[idx], y[idx]
            grad = -(2 / len(idx)) * (Xb.T @ (yb - Xb @ w))
            w = w - lr * grad
        epoch_losses.append(np.mean((y - X @ w) ** 2))
    return w, epoch_losses


best_loss = np.mean((y_train - Xd_train @ w_exact) ** 2)
fig, ax = plt.subplots(figsize=(9, 4.5))
for b in [len(y_train), 32, 1]:
    w_sgd, epoch_losses = minibatch_sgd(Xd_train, y_train, lr=0.01, batch_size=b, n_epochs=40)
    steps = int(np.ceil(len(y_train) / b))
    ax.plot(range(1, 41), epoch_losses, "o-", ms=3, label=f"batch = {b} ({steps} steps/epoch)")
    print(f"batch {b:>3d}: final loss {epoch_losses[-1]:.4f}   (exact minimum {best_loss:.4f})")
ax.axhline(best_loss, color="black", ls=":", label="normal-equation minimum")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("training MSE")
ax.legend()
plt.show()

**Model answer:**

Per **epoch**, smaller batches win by a mile. Full-batch GD takes one step per epoch and is still far from the minimum after 40 epochs, while batch 32 (10 steps per epoch) and batch 1 (313 steps per epoch) get close within a few epochs. The cost is noise: with batch size 1, each step follows a single car's gradient, so the loss jitters and hovers slightly *above* the true minimum instead of settling on it. In practice, a decaying learning rate or a moderate batch size (32 to 256) balances speed and stability. This, not full-batch GD, is how large models are actually trained.

## Part 8: Wrap-Up

In 4 to 5 sentences, connect this week: how do the gradient (calculus), $X^TX$ and its eigenvalues (linear algebra), and the MSE loss (statistics) work together when a model "learns"?

**Model answer:**

The model's error is measured with a statistic, the mean squared error: the average squared deviation from Monday, now between predictions and truth. Predictions are a matrix-vector product $X\vec{w}$, and the gradient of the MSE is $-\frac{2}{m}X^T(\vec{y} - X\vec{w})$, which is calculus expressed in linear algebra. Gradient descent repeatedly steps against that gradient. Its speed and stability depend on the eigenvalues of $X^TX$: the largest caps the learning rate, and the ratio of largest to smallest (made worse by collinear, unscaled features) decides how many steps it takes. Setting the gradient to zero directly gives back Friday's normal equations, so both days solve the same problem by two routes.

> Submit your completed notebook to the Kanban board under your name by end of day.